<a href="https://colab.research.google.com/github/itsvibinraj/AI-Resume-Builder/blob/main/Model_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-Coder-1.5B"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the model and map it to the available device (GPU/CPU)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 # Use float16 to reduce memory overhead
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:

# Freeze all parameters in the LLM
for param in model.parameters():
    param.requires_grad = False

# Put the model in evaluation mode
model.eval()
print("Model successfully loaded and frozen.")

Model successfully loaded and frozen.


In [ ]:
def generate_code(prompt: str, decoding_config: dict) -> str:
    """
    Generates code by applying dynamic decoding parameters to a frozen LLM.
    """
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Extract predicted parameters from the configuration dictionary
    temp = decoding_config.get("temperature", 0.7)
    top_p = decoding_config.get("top_p", 1.0)
    top_k = decoding_config.get("top_k", 50)
    min_p = decoding_config.get("min_p", 0.0)

    # Generate the output using the dynamic parameters
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=max(temp, 0.01),         # Safety check: prevents crash if temp is 0.0
            top_p=top_p,
            top_k=top_k,
            min_p=min_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Slice the generated output to remove the original prompt tokens
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_code = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Truncate extra boilerplate if the model hallucinates beyond the function scope
    stop_sequences = [
        "\ndef ",
        "\nif __name__",
        "\nclass ",
        "\n# Example",
        "\ndef main("
    ]

    for stop_seq in stop_sequences:
        if stop_seq in generated_code:
            generated_code = generated_code.split(stop_seq)[0]

    return generated_code

In [ ]:
# Define contrasting decoding configurations to test
test_configs = {
    "Config 1: Strict / Low Randomness": {
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.05
    },
    "Config 2: Balanced / Standard": {
        "temperature": 0.6,
        "top_p": 0.90,
        "top_k": 50,
        "min_p": 0.0
    },
    "Config 3: High Exploration / Diverse": {
        "temperature": 1.2,
        "top_p": 0.98,
        "top_k": 100,
        "min_p": 0.0
    },
    "Config 4: High Temp with Min-p Truncation": {
        "temperature": 1.2,
        "top_p": 1.0,
        "top_k": 0,
        "min_p": 0.1
    }
}

# Define a HumanEval style test prompt
test_prompt = '''def find_prime_factors(n: int) -> list[int]:
    """Return a list of prime factors of n in ascending order."""
'''

print(f"--- Prompt ---\n{test_prompt.strip()}\n" + "="*50)

# Run the test across all configurations
for name, cfg in test_configs.items():
    print(f"\n[Running {name}]")
    print(f"Parameters: {cfg}")

    # Call your previously defined generation function
    generated_text = generate_code(test_prompt, cfg)

    print("Result:")
    print(generated_text.strip())
    print("-" * 50)

--- Prompt ---
def find_prime_factors(n: int) -> list[int]:
    """Return a list of prime factors of n in ascending order."""

[Running Config 1: Strict / Low Randomness]
Parameters: {'temperature': 0.1, 'top_p': 0.95, 'top_k': 20, 'min_p': 0.05}
Result:
factors = []
    divisor = 2
    while divisor * divisor <= n:
        while n % divisor == 0:
            factors.append(divisor)
            n //= divisor
        divisor += 1
    if n > 1:
        factors.append(n)
    return factors


def main():
    n = int(input("Enter a number: "))
    factors = find_prime_factors(n)
    print(f"The prime factors of {n} are: {factors}")


if __name__ == "__main__":
    main()
--------------------------------------------------

[Running Config 2: Balanced / Standard]
Parameters: {'temperature': 0.6, 'top_p': 0.9, 'top_k': 50, 'min_p': 0.0}
Result:
i = 2
    factors = []
    while i * i <= n:
        if n % i:
            i += 1
        else:
            n //= i
            factors.append(i)
    i

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import threading

# 1. Authenticate ngrok
# Replace with your actual ngrok auth token
ngrok.set_auth_token("3KuCdftbrfafkPiBhMLPQrLNC2R_4wCunztydUvejwhyHZiW1")

# 2. Define the Request Data Model
class GenerationRequest(BaseModel):
    prompt: str
    temperature: float = 0.7
    top_p: float = 1.0
    top_k: int = 50
    min_p: float = 0.0

# 3. Initialize FastAPI App
app = FastAPI()

# 4. Create the API Endpoint
@app.post("/generate")
async def api_generate_code(req: GenerationRequest):
    # Convert the request parameters into the dictionary format your function expects
    decoding_config = {
        "temperature": req.temperature,
        "top_p": req.top_p,
        "top_k": req.top_k,
        "min_p": req.min_p
    }

    # Call the generate_code function you built earlier
    # Make sure your generate_code function and the Qwen model are already loaded in Colab!
    result = generate_code(req.prompt, decoding_config)

    return {"generated_code": result}

# 5. Expose the API to the public internet using ngrok
ngrok_tunnel = ngrok.connect(
    addr="8000",
)
print(f"\n✅ SUCCESS! Share this URL with Abhijith: {ngrok_tunnel.public_url}\n")

# 6. Run the FastAPI server in the background
nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()


✅ SUCCESS! Share this URL with Abhijith: https://large-enhance-viscous.ngrok-free.dev



INFO:     Started server process [6011]


In [ ]:
from pyngrok import ngrok

# Ensure this matches the port configured for your Uvicorn server (e.g., 8000)
public_url = ngrok.connect(8000).public_url
print("API URL:", public_url)

API URL: https://large-enhance-viscous.ngrok-free.dev


In [ ]:

import requests

API_URL = "https://large-enhance-viscous.ngrok-free.dev/generate"

response = requests.post(
    API_URL,
    headers={"ngrok-skip-browser-warning": "1"},
    json={
        "prompt": "def add(a, b):\n    ",
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.05
    },
    timeout=180
)

print("Status:", response.status_code)
print("Response:", response.text[:1000])

INFO:     136.85.167.210:0 - "POST /generate HTTP/1.1" 200 OK
Status: 200
Response: {"generated_code":" print \"ADDING %d + %d\" % (a, b)\n     return a + b\n"}


In [ ]:
import requests
# Testing locally inside Colab
res = requests.post("http://127.0.0.1:8000/generate", json={
    "prompt": "def add(a, b):\n", "temperature": 0.1, "top_p": 0.95, "top_k": 20, "min_p": 0.05
})
print(res.status_code)

INFO:     127.0.0.1:55102 - "POST /generate HTTP/1.1" 200 OK
200


In [ ]:
from pyngrok import ngrok

# Disconnect any old tunnels
ngrok.kill()

# Connect WITHOUT the domain parameter
ngrok_tunnel = ngrok.connect(8000)
print(f"\n✅ SUCCESS! Share this exact URL: {ngrok_tunnel.public_url}/generate\n")


✅ SUCCESS! Share this exact URL with Abhijith: https://large-enhance-viscous.ngrok-free.dev/generate



In [ ]:
ngrok_tunnel = ngrok.connect(8000)
print(ngrok_tunnel.public_url)

https://large-enhance-viscous.ngrok-free.dev
